In [12]:
import requests
import pandas as pd

from IPython.display import Image, display


In [6]:
API_KEY = "gdC7a30RSxIzBrQw15nh2bwppmFZQdkUA2v1f65l"

In [7]:
# Endpoint APOD
url_apod = "https://api.nasa.gov/planetary/apod"

params_apod = {
    "api_key": API_KEY,
    "date": "2024-01-01"  # facultatif ; sinon aujourd'hui
}

response = requests.get(url_apod, params=params_apod)
data_apod = response.json()

# Affichage des informations
print("Titre :", data_apod.get("title"))
print("Date  :", data_apod.get("date"))
print("Explication :", data_apod.get("explanation"))

# Affichage de l'image (sans matplotlib)
if data_apod.get("media_type") == "image":
    display(Image(url=data_apod["url"]))
else:
    print("Média du jour :", data_apod.get("url"))

Titre : NGC 1232: A Grand Design Spiral Galaxy
Date  : 2024-01-01
Explication : Galaxies are fascinating not only for what is visible, but for what is invisible. Grand spiral galaxy NGC 1232, captured in detail by one of the Very Large Telescopes, is a good example.  The visible is dominated by millions of bright stars and dark dust, caught up in a gravitational swirl of spiral arms revolving about the center. Open clusters containing bright blue stars can be seen sprinkled along these spiral arms, while dark lanes of dense interstellar dust can be seen sprinkled between them. Less visible, but detectable, are billions of dim normal stars and vast tracts of interstellar gas, together wielding such high mass that they dominate the dynamics of the inner galaxy.  Leading theories indicate that even greater amounts of matter are invisible, in a form we don't yet know. This pervasive dark matter is postulated, in part, to explain the motions of the visible matter in the outer regions of gal

In [9]:
url_neo = "https://api.nasa.gov/neo/rest/v1/feed"

params_neo = {
    "api_key": API_KEY,
    "start_date": "2024-01-01",
    "end_date": "2024-01-07"   # max 7 jours d'écart
}

response_neo = requests.get(url_neo, params=params_neo)
data_neo = response_neo.json()

# Aperçu de la structure
print(data_neo.keys())
print("Nombre d'astéroïdes :", data_neo["element_count"])



dict_keys(['links', 'element_count', 'near_earth_objects'])
Nombre d'astéroïdes : 45


In [13]:
# La structure est : data_neo['near_earth_objects'][date] -> liste d'astéroïdes
rows = []

for date, asteroids in data_neo["near_earth_objects"].items():
    for ast in asteroids:
        # Diamètre min estimé (km)
        diameter_min_km = ast["estimated_diameter"]["kilometers"]["estimated_diameter_min"]

        # Vitesse relative (km/s) — on prend la première mesure disponible
        velocity_km_s = float(
            ast["close_approach_data"][0]["relative_velocity"]["kilometers_per_second"]
        )

        rows.append({
            "ID de l'astéroïde": ast["id"],
            "Nom de l'astéroïde": ast["name"],
            "Diamètre minimal estimé (km)": diameter_min_km,
            "Magnitude absolue": ast["absolute_magnitude_h"],
            "Vitesse relative (km/s)": velocity_km_s
        })

df = pd.DataFrame(rows)
print(df.head())
print(df.shape)


  ID de l'astéroïde  Nom de l'astéroïde  Diamètre minimal estimé (km)  \
0           2415949  415949 (2001 XY10)                      0.355267   
1           3160747         (2003 SR84)                      0.016771   
2           3309828         (2005 YQ96)                      0.199781   
3           3457842         (2009 HC21)                      0.101054   
4           3553062         (2010 XA11)                      0.016016   

   Magnitude absolue  Vitesse relative (km/s)  
0              19.37                15.890526  
1              26.00                10.719182  
2              20.62                15.670282  
3              22.10                 6.080866  
4              26.10                 8.741383  
(45, 5)


In [14]:
df.to_csv("asteroides_neows.csv", index=False, encoding="utf-8")
print("Fichier 'asteroides_neows.csv' exporté avec succès.")

Fichier 'asteroides_neows.csv' exporté avec succès.
